# Imports

In [ ]:
%matplotlib widget
import sys
import os
import mne

sys.path.append(os.path.abspath('../src'))
from B_EEG_Preprocessing.load_raw import load_raw
from B_EEG_Preprocessing.set_montage_and_check import set_montage_and_check
from B_EEG_Preprocessing.basic_filtering import basic_filtering
from B_EEG_Preprocessing.detect_bad_channels import detect_bad_channels
from B_EEG_Preprocessing.detect_leakage_channels import detect_leakage_channels
from B_EEG_Preprocessing.apply_manual_bad_channels import apply_manual_bad_channels
from B_EEG_Preprocessing.get_segments_to_keep import get_segments_to_keep
from B_EEG_Preprocessing.ica_artefact_removal import ica_artefact_removal
from B_EEG_Preprocessing.save_eeg_cleaning_results import save_eeg_cleaning_results
from B_EEG_Preprocessing.ica_artefact_removal import plot_ica_before_after

# Variables

In [ ]:
subjectID = 'Pilote002'
date      = '2026-07-10'
task      = 'Task1_PPS'
data_path =  '/Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data'
file_name = f"{subjectID}_{date}_{task}"
reference_mode = 'average'

### Load Data

In [ ]:
vhdr_file_path = os.path.join(data_path , subjectID, task, 'Raw', 'EEG', file_name + ".vhdr")   
raw = load_raw(vhdr_file_path)

### Set montage

In [ ]:
raw_montage = set_montage_and_check(raw)

### Check bad channels

In [ ]:
_ , bad_report = detect_bad_channels(raw_montage)   # marks raw_filt.info["bads"]

In [ ]:
detect_leakage_channels(raw_montage)

In [ ]:
raw_montage_no_bad_ch = apply_manual_bad_channels(raw_montage, subjectID)

### Basic filtering

In [ ]:
raw_filt = basic_filtering(
    raw_montage_no_bad_ch,
    l_freq=0.1,        # high-pass  (use 1.0 for cleaner ERPs)
    h_freq=80.0,       # low-pass
    notch_freq=50.0,   # power-line (50 Hz in Europe)
    reference=reference_mode    # e.g. "average" for CAR, or a channel name
)

### ICA cleaning

In [ ]:

# Compute the number of components to use for this analysis
n_good_channels = len(mne.pick_types(raw_filt.info, eeg=True, exclude="bads"))
n_components = n_good_channels - 1
segments2keep = get_segments_to_keep(task, subjectID)

raw_clean, ica, excluded, component_labels  = ica_artefact_removal(
    raw_filt,
    ica_segments=segments2keep,
    n_components=n_components,        # let MNE infer from data rank — safest
    method="infomax",       # extended Infomax is set automatically inside the function
    label_threshold=0.7, # conservative; lower to 0.7 if you feel artefacts are slipping through
    montage=raw_filt.get_montage()
)

In [ ]:
fig = plot_ica_before_after(
    raw_before=raw_filt,
    raw_after=raw_clean,
    tmin=40,
    tmax=100,
    n_channels=10,
    random_state=42
)

In [ ]:
raw_clean.plot(duration=180, start=60, n_channels=20, block=True);

In [ ]:
raw_clean.compute_psd(fmax=150).plot();

### Save clean signals

In [ ]:
save_eeg_cleaning_results(
    file_path=os.path.join(data_path, subjectID, task,'Output','clean_EEG'),
    file_name=f"{subjectID}_{date}_{task}",
    raw_clean=raw_clean,
    ica=ica
)